In [ ]:
# Model Evaluation and Comparison
# This notebook evaluates both RAC and Baseline models on the TaoWu test set.

# **Purpose:**
# - Compare RAC (retrieval-augmented) vs Baseline (ab-initio) performance
# - Calculate accuracy, AUC-ROC, confusion matrices
# - Identify cases where RAC outperforms Baseline
# - Provide detailed per-subject predictions

import os
import torch
import pandas as pd
import numpy as np
import faiss  # For retrieval in RAC model
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from models import GraphAutoencoder, AttentionMechanism, RAC_Model, Baseline_GNN
from utils import FCDataset

# ===== DEFINE FILE PATHS =====
output_dir = r'C:\Users\danie\Documents\Projects\Master thesis\Tara\Scripts\Outputs'
master_csv = os.path.join(output_dir, "master_metadata.csv")
kb_embeddings_path = os.path.join(output_dir, 'kb_embeddings.npy')  # Knowledge base vectors
index_path = os.path.join(output_dir, 'knowledge_base.index')  # FAISS index

# ===== MODEL WEIGHT PATHS =====
# Weights from previous training scripts
encoder_weights = os.path.join(output_dir, 'gae_encoder.pth')  # Pre-trained encoder (Script 02)
rac_weights = os.path.join(output_dir, 'rac_model.pth')  # Trained RAC model (Script 04)
baseline_weights = os.path.join(output_dir, 'baseline_model.pth')  # Trained baseline (Script 05)

# ===== PREPARE TEST DATA =====
# Load master metadata
df = pd.read_csv(master_csv)

# Extract TaoWu dataset as target
df_target = df[df['dataset_source'] == 'TaoWu'].reset_index(drop=True)

# ===== CREATE TEST SET =====
# Use same random_state=42 as Scripts 04 and 05
# Scripts 04/05 used 80% for train/val, we now use the held-out 20% for testing
_, df_test = train_test_split(df_target, test_size=0.2, stratify=df_target['label'], random_state=42)

# Create test dataloader
# batch_size=1 allows individual subject analysis
# shuffle=False maintains order for tracking
test_loader = DataLoader(FCDataset(df_test), batch_size=1, shuffle=False)

# ===== LOAD RAC MODEL =====
# Load FAISS index for nearest neighbor retrieval
index = faiss.read_index(index_path)

# Load knowledge base embeddings
kb_embeddings = np.load(kb_embeddings_path).astype('float32')

# ===== INITIALIZE AND LOAD RAC STRUCTURE =====
# Initialize base GraphAutoencoder with pre-trained weights
base_gae = GraphAutoencoder(100, 100, 64, 128)
base_gae.encoder.load_state_dict(torch.load(encoder_weights))

# Initialize attention mechanism
attention = AttentionMechanism(embedding_dim=128)

# Initialize RAC model (encoder + attention + classifier)
rac_model = RAC_Model(base_gae.encoder, attention, embedding_dim=128)

# Load trained RAC weights from Script 04
rac_model.load_state_dict(torch.load(rac_weights))

# Set to evaluation mode (disables dropout, etc.)
rac_model.eval()

# ===== LOAD BASELINE MODEL =====
# Initialize a fresh GraphAutoencoder (baseline used random initialization)
fresh_gae = GraphAutoencoder(100, 100, 64, 128)

# Initialize Baseline_GNN with the fresh encoder
baseline_model = Baseline_GNN(gae_encoder=fresh_gae.encoder, embedding_dim=128)

# Load trained baseline weights from Script 05
baseline_model.load_state_dict(torch.load(baseline_weights))

# Set to evaluation mode
baseline_model.eval()

# ===== INITIALIZE RESULT STORAGE =====
all_labels = []  # Ground truth labels
rac_probs = []  # RAC predicted probabilities
baseline_probs = []  # Baseline predicted probabilities

# ===== EVALUATION LOOP =====
print("Starting Evaluation on TaoWu Test Set...")

# Disable gradient computation for efficiency
with torch.no_grad():
    for data, label in test_loader:
        # ===== A. RAC PREDICTION (WITH RETRIEVAL) =====
        # Step 1: Generate query embedding using pre-trained encoder
        v_query = rac_model.gae_encoder(data.x, data.edge_index, data.edge_weight, data.batch)
        
        # Step 2: Retrieve 10 nearest neighbors from knowledge base
        distances, indices = index.search(v_query.cpu().numpy().astype('float32'), k=10)
        
        # Step 3: Get actual embedding vectors for retrieved neighbors
        v_retrieved = torch.from_numpy(kb_embeddings[indices]).to(v_query.device)
        
        # Step 4: Forward pass through RAC model (uses attention over retrieved neighbors)
        r_pred, _ = rac_model(data.x, data.edge_index, data.edge_weight, data.batch, v_retrieved)
        
        # ===== B. BASELINE PREDICTION (NO RETRIEVAL) =====
        # Direct forward pass through baseline model
        b_pred = baseline_model(data.x, data.edge_index, data.edge_weight, data.batch)
        
        # ===== C. STORE RESULTS =====
        all_labels.append(label.item())  # Ground truth (0 or 1)
        rac_probs.append(r_pred.item())  # RAC probability (0 to 1)
        baseline_probs.append(b_pred.item())  # Baseline probability (0 to 1)

# ===== POST-PROCESS RESULTS =====
# Convert probabilities to binary predictions (threshold = 0.5)
rac_classes = [1 if p > 0.5 else 0 for p in rac_probs]
baseline_classes = [1 if p > 0.5 else 0 for p in baseline_probs]

# ===== CALCULATE AND PRINT METRICS =====
def print_metrics(name, labels, probs, classes):
    """Helper function to print comprehensive metrics"""
    print(f"\n--- {name} Results ---")
    print(f"Accuracy:  {accuracy_score(labels, classes):.4f}")  # Overall accuracy
    print(f"AUC-ROC:   {roc_auc_score(labels, probs):.4f}")  # Area under ROC curve
    print("Confusion Matrix:")
    print(confusion_matrix(labels, classes))  # [[TN, FP], [FN, TP]]

# Print metrics for both models
print_metrics("RAC MODEL (Retrieval)", all_labels, rac_probs, rac_classes)
print_metrics("BASELINE MODEL (Ab-Initio)", all_labels, baseline_probs, baseline_classes)

# ===== SAVE DETAILED TEST RESULTS =====
results_df = pd.DataFrame({
    'subject_id': df_test['subject_id'].values,  # Subject identifier
    'original_master_index': df_test.index.values,  # Index in master CSV
    'ground_truth': all_labels,  # True label (0=Control, 1=PD)
    'rac_prob': rac_probs,  # RAC predicted probability
    'rac_predicted_class': rac_classes,  # RAC binary prediction
    'baseline_prob': baseline_probs,  # Baseline predicted probability
    'baseline_predicted_class': baseline_classes  # Baseline binary prediction
})

# ===== IDENTIFY RAC ADVANTAGES =====
# Mark cases where RAC was correct but Baseline was wrong
# These are "success stories" demonstrating the value of retrieval
results_df['rac_advantage'] = (results_df['rac_predicted_class'] == results_df['ground_truth']) & \
                             (results_df['baseline_predicted_class'] != results_df['ground_truth'])

# Save results to CSV
results_csv_path = os.path.join(output_dir, 'test_evaluation_results.csv')
results_df.to_csv(results_csv_path, index=False)

# Print summary
print(f"\nDetailed results saved to: {results_csv_path}")
print(f"Number of cases where RAC outperformed Baseline: {results_df['rac_advantage'].sum()}")

In [1]:
import os
import torch
import pandas as pd
import numpy as np
import faiss
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from models import GraphAutoencoder, AttentionMechanism, RAC_Model, Baseline_GNN
from utils import FCDataset

In [2]:
output_dir = r'C:\Users\danie\Documents\Projects\Master thesis\Tara\Scripts\Outputs'
master_csv = os.path.join(output_dir, "master_metadata.csv")
kb_embeddings_path = os.path.join(output_dir, 'kb_embeddings.npy')
index_path = os.path.join(output_dir, 'knowledge_base.index')

# Weights from Scripts 02, 04, and 05
encoder_weights = os.path.join(output_dir, 'gae_encoder.pth')
rac_weights = os.path.join(output_dir, 'rac_model.pth')
baseline_weights = os.path.join(output_dir, 'baseline_model.pth')

In [3]:
# 2. PREPARE TEST DATA
df = pd.read_csv(master_csv)
df_target = df[df['dataset_source'] == 'TaoWu'].reset_index(drop=True)

# We use the same random_state=42. 
# Script 04/05 used 80% for train/val. Now we use the 20% "Test" set.
_, df_test = train_test_split(df_target, test_size=0.2, stratify=df_target['label'], random_state=42)
test_loader = DataLoader(FCDataset(df_test), batch_size=1, shuffle=False) # Batch 1 for individual analysis

In [4]:
# 3. LOAD RAC MODEL
index = faiss.read_index(index_path)
kb_embeddings = np.load(kb_embeddings_path).astype('float32')

# Initialize RAC structure
base_gae = GraphAutoencoder(100, 100, 64, 128)
base_gae.encoder.load_state_dict(torch.load(encoder_weights))
attention = AttentionMechanism(embedding_dim=128)
rac_model = RAC_Model(base_gae.encoder, attention, embedding_dim=128)
rac_model.load_state_dict(torch.load(rac_weights))
rac_model.eval()

RAC_Model(
  (gae_encoder): GAEEncoder(
    (conv1): GCNConv(100, 64)
    (conv2): GCNConv(64, 64)
    (lin_encode): Linear(in_features=64, out_features=128, bias=True)
  )
  (attention_model): AttentionMechanism()
  (classification_head): Sequential(
    (0): Linear(in_features=256, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
    (4): Sigmoid()
  )
)

In [5]:
# 4. Load the trained Baseline model
#    - (Initialize Baseline_GNN, load 'baseline_model.pth')
#    - baseline_model.eval()

# Note: Baseline used a 'fresh' encoder, so we load its specific weights
fresh_gae = GraphAutoencoder(100, 100, 64, 128)
baseline_model = Baseline_GNN(gae_encoder=fresh_gae.encoder, embedding_dim=128)
baseline_model.load_state_dict(torch.load(baseline_weights))
baseline_model.eval()

Baseline_GNN(
  (gae_encoder): GAEEncoder(
    (conv1): GCNConv(100, 64)
    (conv2): GCNConv(64, 64)
    (lin_encode): Linear(in_features=64, out_features=128, bias=True)
  )
  (classification_head): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
    (4): Sigmoid()
  )
)

In [7]:
all_labels = []
rac_probs = []
baseline_probs = []

# 6. Start the evaluation loop
#    - with torch.no_grad():
#    -   for (data, label) in test_dataloader:
#    -     # ... 1. Get RAC prediction (requires retrieval step)
#    -     #      rac_pred = rac_model(...)
#    -     # ... 2. Get Baseline prediction
#    -     #      baseline_pred = baseline_model(data)
#    -     # ... 3. Store results
#    -     #      all_labels.append(label)
#    -     #      rac_predictions.append(rac_pred)
#    -     #      baseline_predictions.append(baseline_pred)

print("Starting Evaluation on TaoWu Test Set...")
with torch.no_grad():
    for data, label in test_loader:
        # A. RAC Prediction (Retrieval Step)
        v_query = rac_model.gae_encoder(data.x, data.edge_index, data.edge_weight, data.batch)
        distances, indices = index.search(v_query.cpu().numpy().astype('float32'), k=10)
        v_retrieved = torch.from_numpy(kb_embeddings[indices]).to(v_query.device)
        
        r_pred, _ = rac_model(data.x, data.edge_index, data.edge_weight, data.batch, v_retrieved)
        
        # B. Baseline Prediction
        b_pred = baseline_model(data.x, data.edge_index, data.edge_weight, data.batch)
        
        # C. Store Results
        all_labels.append(label.item())
        rac_probs.append(r_pred.item())
        baseline_probs.append(b_pred.item())

# 7. Post-process results (convert logits to probabilities/classes)
#    - ...

rac_classes = [1 if p > 0.5 else 0 for p in rac_probs]
baseline_classes = [1 if p > 0.5 else 0 for p in baseline_probs]

# 8. Calculate and print metrics
#    - (e.g., from sklearn.metrics import accuracy_score, roc_auc_score)
#    - print("--- RAC Model Results ---")
#    - print(f"Accuracy: {accuracy_score(all_labels, rac_classes)}")
#    - print(f"AUC: {roc_auc_score(all_labels, rac_probs)}")
#    -
#    - print("--- Baseline Model Results ---")
#    - print(f"Accuracy: {accuracy_score(all_labels, baseline_classes)}")
#    - print(f"AUC: {roc_auc_score(all_labels, baseline_probs)}")

def print_metrics(name, labels, probs, classes):
    print(f"\n--- {name} Results ---")
    print(f"Accuracy:  {accuracy_score(labels, classes):.4f}")
    print(f"AUC-ROC:   {roc_auc_score(labels, probs):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(labels, classes))

print_metrics("RAC MODEL (Retrieval)", all_labels, rac_probs, rac_classes)
print_metrics("BASELINE MODEL (Ab-Initio)", all_labels, baseline_probs, baseline_classes)

# 9. (Optional) Save test results to a file

# 9. SAVE TEST RESULTS TO FILE
results_df = pd.DataFrame({
    'subject_id': df_test['subject_id'].values,
    'original_master_index': df_test.index.values,
    'ground_truth': all_labels,
    'rac_prob': rac_probs,
    'rac_predicted_class': rac_classes,
    'baseline_prob': baseline_probs,
    'baseline_predicted_class': baseline_classes
})

# Identify "Success Stories": Cases where RAC was right and Baseline was wrong
results_df['rac_advantage'] = (results_df['rac_predicted_class'] == results_df['ground_truth']) & \
                             (results_df['baseline_predicted_class'] != results_df['ground_truth'])

results_csv_path = os.path.join(output_dir, 'test_evaluation_results.csv')
results_df.to_csv(results_csv_path, index=False)

print(f"\nDetailed results saved to: {results_csv_path}")
print(f"Number of cases where RAC outperformed Baseline: {results_df['rac_advantage'].sum()}")

Starting Evaluation on TaoWu Test Set...

--- RAC MODEL (Retrieval) Results ---
Accuracy:  0.7500
AUC-ROC:   0.8125
Confusion Matrix:
[[3 1]
 [1 3]]

--- BASELINE MODEL (Ab-Initio) Results ---
Accuracy:  0.7500
AUC-ROC:   0.7500
Confusion Matrix:
[[3 1]
 [1 3]]

Detailed results saved to: C:\Users\danie\Documents\Projects\Master thesis\Tara\Scripts\Outputs\test_evaluation_results.csv
Number of cases where RAC outperformed Baseline: 1
